In [1]:
import numpy as np
import scipy.sparse as sp
import scipy.stats as stats
import matplotlib.pyplot as plt
import pandas as pd
import os
import gc
import zipfile
from numba import njit
from tqdm import tqdm

# ==========================================
# 1. ネットワークと単体複体生成 (Numba JIT)
# ==========================================
@njit
def build_simplicial_complex(N, m, seed=42):
    np.random.seed(seed)
    num_edges = 2 * m * N
    row = np.empty(num_edges, dtype=np.int32)
    col = np.empty(num_edges, dtype=np.int32)
    targets = np.empty(num_edges, dtype=np.int32)
    max_faces = m * m * N
    faces = np.empty((max_faces, 3), dtype=np.int32)

    idx, target_idx, face_idx = 0, 0, 0

    for i in range(m):
        for j in range(m):
            if i != j:
                row[idx] = i; col[idx] = j; idx += 1
                targets[target_idx] = i; target_idx += 1

    if m >= 3:
        faces[face_idx, 0] = 0; faces[face_idx, 1] = 1; faces[face_idx, 2] = 2; face_idx += 1

    chosen = np.empty(m, dtype=np.int32)

    for i in range(m, N):
        c_idx = 0
        while c_idx < m:
            r = np.random.randint(0, target_idx)
            node = targets[r]
            already = False
            for k in range(c_idx):
                if chosen[k] == node:
                    already = True; break
            if not already:
                chosen[c_idx] = node; c_idx += 1

        for j in range(m):
            for k in range(j + 1, m):
                faces[face_idx, 0] = i
                faces[face_idx, 1] = chosen[j]
                faces[face_idx, 2] = chosen[k]
                face_idx += 1

        for k in range(m):
            target = chosen[k]
            row[idx] = i; col[idx] = target; idx += 1
            row[idx] = target; col[idx] = i; idx += 1
            targets[target_idx] = i; target_idx += 1
            targets[target_idx] = target; target_idx += 1

    return row[:idx], col[:idx], faces[:face_idx]

@njit
def evaluate_faces(X, faces, active_faces, theta, theta_safe):
    ruptured = 0
    safe_local_faces = 0
    for f in range(faces.shape[0]):
        i, j, k = faces[f, 0], faces[f, 1], faces[f, 2]
        frustration = max(X[i], X[j], X[k]) - min(X[i], X[j], X[k])
        if active_faces[f]:
            if frustration > theta:
                active_faces[f] = False
                ruptured += 1
            elif frustration < theta_safe:
                safe_local_faces += 1
    return ruptured, safe_local_faces

# ==========================================
# 2. シミュレーション（副作用・Betti 2 追跡モデル）
# ==========================================
def run_side_effect_simulation(N, m, params, seed=42):
    np.random.seed(seed)
    dt = params['dt']
    T1, T2, T3, T4 = params['T1'], params['T2'], params['T3'], params['T4']

    row, col, faces = build_simplicial_complex(N, m, seed)
    data = np.ones(len(row), dtype=np.float32)
    A_sparse = sp.csr_matrix((data, (row, col)), shape=(N, N))
    degrees = np.array(A_sparse.sum(axis=1)).flatten()
    degrees[degrees == 0] = 1.0
    hub_node = np.argmax(degrees)

    del row, col, data; gc.collect()

    X = np.random.uniform(0, params['epsilon'], N).astype(np.float32)
    b = np.zeros(N, dtype=np.float32)

    num_faces = faces.shape[0]
    active_faces = np.ones(num_faces, dtype=bool)

    history_betti_1 = []
    history_betti_2 = [] # カルト化指数
    history_ks_stat = []

    total_ruptured = 0
    total_injected_b1 = 0
    total_betti_2_generated = 0
    X_prev = np.copy(X)

    for step in tqdm(range(T4), desc=f"Simulating N={N}", leave=False):
        if step == T1:
            b[hub_node] = 0.9
        elif step == T2:
            b[hub_node] = 0.0
        elif step == T3:
            # 高次元介入: 破裂した穴(β1)を埋めるために2-単体を過剰気味に注入する
            # ここでは「完全な回復を狙って100%相当の2-単体を投下」したとする
            M_prime = int(total_ruptured * 1.0)
            total_injected_b1 += M_prime

            # 副作用（カルト化＝β2）の生成:
            # 局所的に密集して注入された面同士が四面体(3-単体)を形成する。
            # 注入量(M_prime)の2乗に比例して閉鎖的なボイド(β2)が急増する近似モデル
            _, safe_count = evaluate_faces(X, faces, active_faces, params['theta'], params['theta_safe'])
            # 局所密度によるカルト形成ペナルティ
            kappa = params['kappa_cult']
            new_betti_2 = int(kappa * (M_prime ** 2) / max(1, safe_count))
            total_betti_2_generated += new_betti_2

        P = A_sparse.dot(X) / degrees
        noise = np.random.normal(0, 1, N).astype(np.float32) * np.sqrt(dt)
        dX = (P + b - X) * dt + params['sigma'] * noise
        X = np.clip(X + dX, 0, 1)

        new_ruptured, _ = evaluate_faces(X, faces, active_faces, params['theta'], params['theta_safe'])
        total_ruptured += new_ruptured

        current_betti_1 = max(0, total_ruptured - total_injected_b1)
        history_betti_1.append(current_betti_1 / max(1, num_faces))

        # Betti 2 (カルト化リスク) も正規化して記録
        history_betti_2.append(total_betti_2_generated / max(1, num_faces))

        if step <= T1 and step % 10 == 0:
            ks_stat, _ = stats.ks_2samp(X[::max(1, N//1000)], X_prev[::max(1, N//1000)])
            history_ks_stat.append((step, ks_stat))
            X_prev = np.copy(X)

    return history_betti_1, history_betti_2, history_ks_stat

# ==========================================
# 3. メイン実行スクリプト
# ==========================================
def main():
    N_list = [1000, 10000, 100000, 1000000]

    params = {
        'dt': 0.1,
        'T1': 100,
        'T2': 200,
        'T3': 300,
        'T4': 400,
        'epsilon': 0.05,
        'sigma': 0.02,
        'theta': 0.25,
        'theta_safe': 0.15,
        'kappa_cult': 5.0 # カルト形成の非線形係数
    }

    output_dir = "Advanced_SideEffect_Betti2"
    os.makedirs(output_dir, exist_ok=True)
    summary_data = []

    for N in N_list:
        try:
            print(f"\n[{N:,} Nodes] カルト化（β2）副作用・自己組織化検証開始...")
            betti1_hist, betti2_hist, ks_hist = run_side_effect_simulation(N, 3, params)

            # 1. KS検定による自己組織化証明
            steps_ks, ks_vals = zip(*ks_hist)
            plt.figure(figsize=(8, 5))
            plt.plot(steps_ks, ks_vals, 'g-o', linewidth=2)
            plt.title(f'Convergence to Empirical Stationary Distribution (N={N})')
            plt.xlabel('Burn-in Steps')
            plt.ylabel('KS Statistic (Distribution Drift)')
            plt.grid(True, alpha=0.3)
            plt.savefig(os.path.join(output_dir, f'Adv_SideEffect_KS_N{N}.png'), dpi=300)
            plt.close()

            # 2. 治療のトレードオフ（β1減少 vs β2急増）のプロット
            df = pd.DataFrame({
                'TimeStep': np.arange(params['T4']),
                'Betti_1_Holes': betti1_hist,
                'Betti_2_Cults': betti2_hist
            })
            df.to_csv(os.path.join(output_dir, f'Adv_side_effect_N{N}.csv'), index=False)

            fig, ax1 = plt.subplots(figsize=(12, 6))

            ax1.set_xlabel('Time Step')
            ax1.set_ylabel('Targeted Defect ($\Delta \\beta_1 / |F|$)', color='green')
            ax1.plot(df['TimeStep'], df['Betti_1_Holes'], 'g-', linewidth=2.5, label='Betti 1 (Social Voids)')
            ax1.tick_params(axis='y', labelcolor='green')

            ax2 = ax1.twinx()
            ax2.set_ylabel('Side Effect: Cult Formation ($\Delta \\beta_2 / |F|$)', color='purple')
            ax2.plot(df['TimeStep'], df['Betti_2_Cults'], 'purple', linestyle='--', linewidth=2.5, label='Betti 2 (Cult Echo-Chambers)')
            ax2.tick_params(axis='y', labelcolor='purple')

            ax1.axvspan(0, params['T1'], color='gray', alpha=0.1, label='Phase 0: Burn-in')
            ax1.axvspan(params['T1'], params['T2'], color='red', alpha=0.1, label='Phase 1: Pathogenesis')
            ax1.axvspan(params['T2'], params['T3'], color='blue', alpha=0.1, label='Phase 1b: Hysteresis')
            ax1.axvspan(params['T3'], params['T4'], color='orange', alpha=0.2, label='Phase 3: High-D Intervention')

            fig.tight_layout()
            plt.title(f'Therapeutic Trade-off: Healing Voids vs. Generating Cults (N={N:,})')
            plt.savefig(os.path.join(output_dir, f'Adv_SideEffect_Tradeoff_N{N}.png'), dpi=300)
            plt.close()

            summary_data.append({
                'N': N,
                'Betti_1_Final': betti1_hist[-1],
                'Betti_2_Final': betti2_hist[-1]
            })
            gc.collect()

        except MemoryError:
            print(f"[警告] N={N} でメモリ不足。")
            break

    # 3. まとめ出力とZIP化
    if summary_data:
        df_sum = pd.DataFrame(summary_data)
        df_sum.to_csv(os.path.join(output_dir, 'Adv_side_effect_summary.csv'), index=False)

        plt.figure(figsize=(8, 6))
        plt.plot(df_sum['N'], df_sum['Betti_1_Final'], 'g-o', linewidth=2.5, label='Residual Betti 1 (Unhealed Voids)')
        plt.plot(df_sum['N'], df_sum['Betti_2_Final'], 'purple', marker='s', linestyle='--', linewidth=2.5, label='Generated Betti 2 (Cult Structures)')
        plt.xscale('log')
        plt.yscale('log')
        plt.title('Asymptotic Side Effects across Network Scales')
        plt.xlabel('Network Size N (Log Scale)')
        plt.ylabel('Normalized Topological Defect (Log Scale)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(output_dir, 'Adv_side_effect_summary_plot.png'), dpi=300)
        plt.close()

        zip_filename = 'Advanced_Validation_Cult_SideEffect.zip'
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, _, files in os.walk(output_dir):
                for file in files:
                    zipf.write(os.path.join(root, file), arcname=file)
        print(f"\n🎉 副作用検証の計算完了！結果は '{zip_filename}' に出力されました。")

if __name__ == "__main__":
    main()

<>:204: SyntaxWarning: invalid escape sequence '\D'
<>:209: SyntaxWarning: invalid escape sequence '\D'
<>:204: SyntaxWarning: invalid escape sequence '\D'
<>:209: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_2761/3660635814.py:204: SyntaxWarning: invalid escape sequence '\D'
  ax1.set_ylabel('Targeted Defect ($\Delta \\beta_1 / |F|$)', color='green')
/tmp/ipykernel_2761/3660635814.py:209: SyntaxWarning: invalid escape sequence '\D'
  ax2.set_ylabel('Side Effect: Cult Formation ($\Delta \\beta_2 / |F|$)', color='purple')



[1,000 Nodes] カルト化（β2）副作用・自己組織化検証開始...



[10,000 Nodes] カルト化（β2）副作用・自己組織化検証開始...



[100,000 Nodes] カルト化（β2）副作用・自己組織化検証開始...



[1,000,000 Nodes] カルト化（β2）副作用・自己組織化検証開始...



🎉 副作用検証の計算完了！結果は 'Advanced_Validation_Cult_SideEffect.zip' に出力されました。
